# Advanced deep learning, explainability and uncertainty

Owner: P5.

## Goal

Advanced deep learning, explainability and uncertainty

## Inputs

Image loader and baseline from P4; split from P1

## Tasks

- [ ] Compare ConvNeXt, ViT and DINOv2 with a common budget.
- [ ] Compare class-weighted cross-entropy to unweighted training.
- [ ] Implement and inspect Grad-CAM on the CNN path.
- [ ] Calibrate conformal prediction on validation. Audit coverage and set sizes on test.
- [ ] Export the selected image checkpoint and document its loading interface.

## Outputs

- `src/glazevision/explain.py`
- `src/glazevision/uncertainty.py`
- `models/best_image_model.pt`
- `notebooks/05_advanced_dl_explainability.ipynb`

In [ ]:
from glazevision.experiments import run_deep

backbones = ("convnext_tiny", "vit_small_patch16_224", "vit_small_patch14_dinov2")
for backbone in backbones:
    run_deep(backbone, epochs=2, balanced=True, owner="P5")

# TODO(P5): Lock the selected backbone using validation scores before the final test run.
selected = "convnext_tiny"
run_final_test = False
if run_final_test:
    run_deep(selected, epochs=2, balanced=True, owner="P5", export=True,
             conformal=True, evaluation_split="test")

from PIL import Image
import matplotlib.pyplot as plt
from glazevision.config import ROOT
from glazevision.data import joined_samples
from glazevision.image_models import load_image_model
from glazevision.image_data import image_transform
from glazevision.explain import gradcam

if (ROOT / "models/best_image_model.pt").exists():
    sample = joined_samples().query("split == 'val'").dropna(subset=["image_path"]).iloc[0]
    with Image.open(sample.image_path) as photo:
        tensor = image_transform()(photo.convert("RGB")).unsqueeze(0)
    model = load_image_model(ROOT / "models/best_image_model.pt")
    plt.imshow(gradcam(model, tensor), cmap="magma")
    plt.show()

In [ ]:
# TODO(P5): Record actual findings, run settings and limitations after evaluation.